In [40]:
CSV_PATH    = "jobs_for_embeddings.csv"              # path to your jobs CSV file
ES_URL      = "https://localhost:9200" # Elasticsearch URL
ES_USER     = "elastic"
ES_PASS     = "7S6f9I-bCEFQzsLoQ3IE"
SLEEP_SEC   = 0.0                     # optional delay between rows (0 = max speed)
NUM_WORKERS = 8                       # threads for parallel embedding & indexing
BATCH_SIZE  = 100                     # rows per bulk-index batch


## 1 — Imports

In [41]:
import time
import json
import hashlib

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from sentence_transformers import SentenceTransformer
from elasticsearch import Elasticsearch

## 2 — Load model

In [4]:
print("Loading JobBERT — this may take a minute on first run...")
jobbert = SentenceTransformer("TechWolf/JobBERT-v2")
DIM = jobbert.get_embedding_dimension()  
EMBEDDING_DIM = DIM * 3                           
print(f"Model ready. Segment dim: {DIM} | Final embedding dim: {EMBEDDING_DIM}")

Loading JobBERT — this may take a minute on first run...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model ready. Segment dim: 1024 | Final embedding dim: 3072


## 3 — Connect to Elasticsearch & create index

In [12]:
es = Elasticsearch(
    ES_URL,
    basic_auth=(ES_USER, ES_PASS),
    verify_certs=False,
    ssl_show_warn=False,
)

In [16]:
# Delete the Elasticsearch index if it exists
INDEX_NAME = "jobs_strat2_normal"
try:
    if es.indices.exists(index=INDEX_NAME):
        resp = es.indices.delete(index=INDEX_NAME)
        print(f"Deleted index '{INDEX_NAME}': {resp}")
    else:
        print(f"Index '{INDEX_NAME}' does not exist.")
except Exception as e:
    print(f"Error deleting index '{INDEX_NAME}': {e}")

Deleted index 'jobs_strat2_normal': {'acknowledged': True}


In [42]:
INDEX_NAME = "jobs_strat2_normal_actual"

mapping = {
    "mappings": {
        "properties": {
            "job_id":          {"type": "keyword"},
            "normal_job_title":       {"type": "text"},
            "job_description": {"type": "text"},
            "skills":          {"type": "keyword"},
            "embedding": {
                "type":       "dense_vector",
                "dims":       EMBEDDING_DIM,
                "index":      True,
                "similarity": "cosine",
            },
        }
    }
}

try:
    exists = es.indices.exists(index=INDEX_NAME)
except Exception as e:
    print(f"Error checking index existence: {e}")
    if getattr(e, "status_code", None) == 401:
        print("Authentication failed (401). Check ES_URL, ES_USER and ES_PASS.")
    raise

if not exists:
    try:
        resp = es.indices.create(index=INDEX_NAME, body=mapping)
        print(f"Created index '{INDEX_NAME}': {resp}")
    except Exception as e:
        print(f"Error creating index '{INDEX_NAME}': {e}")
        raise
else:
    print(f"Index '{INDEX_NAME}' already exists — skipping creation")

Created index 'jobs_strat2_normal_actual': {'acknowledged': True, 'shards_acknowledged': True, 'index': 'jobs_strat2_normal_actual'}


In [ ]:
# Check whether the Elasticsearch index exists and whether it's empty
InDEX_NAME = "jobs_strat2_normal"
try:
    if not es.indices.exists(index=INDEX_NAME):
        print(f"Index '{INDEX_NAME}' does not exist.")
    else:
        count = es.count(index=INDEX_NAME)["count"]
        if count == 0:
            print(f"Index '{INDEX_NAME}' is empty (0 documents).")
        else:
            print(f"Index '{INDEX_NAME}' contains {count:,} documents.")
except Exception as e:
    print("Error checking index:", str(e))

Index 'jobs_strat2_normal' contains 50 documents.


## 4 — Load & preview CSV

In [43]:
df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df):,} rows")


Loaded 213,848 rows


## 5 — Clean & prepare

In [47]:
def generate_job_embedding(title: str, description: str, skills: list[str]) -> list[float]:
    """
    Concatenate JobBERT encodings of title, description, and skills
    into a single 3072-dim vector: [title(1024) | desc(1024) | skills(1024)]
    """
    title_emb  = jobbert.encode(title       if title       else "unknown")
    desc_emb   = jobbert.encode(description if description else "")
    skills_emb = jobbert.encode(" ".join(skills) if skills else "general")
    return np.concatenate([title_emb, desc_emb, skills_emb]).tolist()


# Apply cleaning
df["_job_id"] = df["job_id"]
df["_title"]   = df["esco_title"]
df["_skills"]  = df["new_skills"].apply(lambda x: [s.strip() for s in str(x).split("|") if s.strip()])
df["_description"] = df["job_description"]

before = len(df)
df = df[df["_title"] != ""].reset_index(drop=True)
print(f"Dropped {before - len(df)} rows with no title. Ready to embed: {len(df):,}")

Dropped 0 rows with no title. Ready to embed: 213,848


## 6 — Dry run (first 3 rows)

In [45]:
for _, row in df.head(3).iterrows():
    emb = generate_job_embedding(row["_title"], row["_description"], row["_skills"])
    print(f"[OK] '{row['_title']}' → {len(emb)} dims | skills: {row['_skills'][:3]}")

[OK] 'automation controls engineer' → 3072 dims | skills: ['sql', 'planning', 'java']
[OK] 'security operations manager' → 3072 dims | skills: ['python', 'linux', 'incident response']
[OK] 'solutions architect' → 3072 dims | skills: ['data warehousing', 'data analysis', 'etl']


## 7 — Bulk embed & index
Embeds and indexes every row directly into Elasticsearch. Errors are collected without stopping the run.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from elasticsearch.helpers import bulk
import threading
import traceback
import time

# Thread-safe counters
_lock   = threading.Lock()
indexed = 0
errors  = []

# Prepare rows to index (do not check ES for existing IDs; re-embed everything)
rows_to_index = [row for _, row in df.iterrows()]
print(f"[DEBUG] Total rows to index: {len(rows_to_index):,}")

if not rows_to_index:
    print("Nothing to do.")
else:
    def embed_and_prepare(row):
        """Embed one row and return an ES action dict (or raise on error)."""
        job_id = row["_job_id"]
        try:
            print(f"[DEBUG] Start embed: {job_id} | title='{row.get('_title')}' | skills={row.get('_skills')}")
            start = time.time()
            embedding = generate_job_embedding(
                row["_title"],
                row["_description"],
                row["_skills"],
            )
            elapsed = time.time() - start
            print(f"[DEBUG] Done embed: {job_id} ({len(embedding)} dims) in {elapsed:.2f}s")
            return {
                "_index": INDEX_NAME,
                "_id":    job_id,
                "_source": {
                    "job_id":            job_id,
                    "normal_job_title":  row["_title"],
                    "job_description":   row["_description"],
                    "skills":            row["_skills"],
                    "embedding":         embedding,
                },
            }
        except Exception as e:
            print(f"[ERROR] embed_and_prepare failed for {job_id}: {e}")
            print(traceback.format_exc())
            raise

    def flush_batch(batch):
        """Bulk-index a list of ES action dicts; return (n_ok, failed_list)."""
        if not batch:
            return 0, []
        print(f"[DEBUG] Flushing batch of {len(batch):,} docs. First id: {batch[0]['_id'] if batch else 'N/A'}")
        ok, failed = bulk(es, batch, raise_on_error=False, stats_only=False)
        print(f"[DEBUG] Bulk returned ok={ok} failed={len(failed) if failed else 0}")
        return ok, failed

    pending_batch = []

    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as pool:
        futures = {pool.submit(embed_and_prepare, row): row["_job_id"] for row in rows_to_index}
        print(f"[DEBUG] Submitted {len(futures):,} tasks to ThreadPoolExecutor (workers={NUM_WORKERS})")
        for future in tqdm(as_completed(futures), total=len(futures), desc="Embedding & indexing"):
            job_id = futures[future]
            try:
                action = future.result()
                batch_to_flush = None
                with _lock:
                    pending_batch.append(action)
                    if len(pending_batch) >= BATCH_SIZE:
                        batch_to_flush = pending_batch[:]
                        pending_batch.clear()
                if batch_to_flush:
                    ok, failed = flush_batch(batch_to_flush)
                    with _lock:
                        indexed += ok
                        if failed:
                            errors.extend(failed if isinstance(failed, list) else [failed])
            except Exception as exc:
                with _lock:
                    err_info = {"job_id": job_id, "error": str(exc), "trace": traceback.format_exc()}
                    errors.append(err_info)
                    print(f"[ERROR] Task failed for {job_id}: {exc}")

    # Flush any remaining docs
    if pending_batch:
        print(f"[DEBUG] Flushing final pending batch of {len(pending_batch):,} docs")
        ok, failed = flush_batch(pending_batch)
        indexed += ok
        if failed:
            errors.extend(failed if isinstance(failed, list) else [failed])

    print(f"\nDone.  Indexed: {indexed:,} | Errors: {len(errors)}")
    if errors:
        print(f"[DEBUG] Sample errors (up to 5): {errors[:5]}")


[DEBUG] Total rows to index: 213,848
[DEBUG] Start embed: 0 | title='automation controls engineer' | skills=['sql', 'planning', 'java', 'selenium', 'etl', 'jenkins', 'unix', 'go', 'object oriented programming', 'test automation']
[DEBUG] Start embed: 1 | title='security operations manager' | skills=['python', 'linux', 'incident response', 'unix']
[DEBUG] Start embed: 2 | title='solutions architect' | skills=['data warehousing', 'data analysis', 'etl', 'business intelligence', 'data lakes']
[DEBUG] Start embed: 3 | title='Java Developer (mid level)- FT- GREAT culture, modern technologies, career growth' | skills=['sql', 'c', 'oracle', 'sql server', 'linux', 'java', 'less', 'unix', 'tcp/ip', 'mysql']
[DEBUG] Start embed: 4 | title='cloud DevOps engineer' | skills=['chef', 'docker', 'linux', 'ansible', 'microservices', 'puppet', 'go', 'devops', 'continuous integration']
[DEBUG] Start embed: 5 | title='ICT system architect' | skills=['nan']
[DEBUG] Start embed: 6 | title='ICT network engin

Embedding & indexing:   0%|          | 0/213848 [00:00<?, ?it/s]

[DEBUG] Done embed: 5 (3072 dims) in 16.85s
[DEBUG] Start embed: 8 | title='IoT developer' | skills=['c', 'javascript']
[DEBUG] Done embed: 1 (3072 dims) in 16.91s
[DEBUG] Start embed: 9 | title='application security engineer' | skills=['javascript', 'linux', 'java', 'incident response', 'jquery', 'http', 'ruby']
[DEBUG] Done embed: 4 (3072 dims) in 16.91s
[DEBUG] Start embed: 10 | title='full stack software engineer' | skills=['python']
[DEBUG] Done embed: 7 (3072 dims) in 16.97s
[DEBUG] Start embed: 11 | title='ICT security administrator' | skills=['sql', 'c', 'operating systems', 'oracle', 'linux', 'java', 'active directory', 'unix']
[DEBUG] Done embed: 6 (3072 dims) in 17.01s
[DEBUG] Start embed: 12 | title='hardware engineer' | skills=['agile', 'listings', 'java', 'http']
[DEBUG] Done embed: 2 (3072 dims) in 17.02s
[DEBUG] Start embed: 13 | title='regional sales manager' | skills=['planning']
[DEBUG] Done embed: 0 (3072 dims) in 17.08s
[DEBUG] Start embed: 14 | title='project mana

## 8 — Inspect errors

In [10]:
if errors:
    err_df = pd.DataFrame(errors)
    display(err_df)
    err_df.to_csv("embedding_errors.csv", index=False)
    print("Saved to embedding_errors.csv")
else:
    print("No errors — all jobs indexed successfully.")

,job_id,title,error
0,https://www.dice.com/jobs/detail/AUTOMATION-TE...,automation controls engineer,'_summary'
1,https://www.dice.com/jobs/detail/Information-S...,security operations manager,'_summary'
2,https://www.dice.com/jobs/detail/Business-Solu...,solutions architect,'_summary'
3,https://www.dice.com/jobs/detail/Java-Develope...,Other,'_summary'
4,https://www.dice.com/jobs/detail/DevOps-Engine...,cloud DevOps engineer,'_summary'
...,...,...,...
216331,https://www.linkedin.com/jobs/view/test-engine...,hardware engineer,'_summary'
216332,https://uk.linkedin.com/jobs/view/fire-securit...,IT infrastructure engineer,'_summary'
216333,https://www.linkedin.com/jobs/view/senior-soft...,Other,'_summary'
216334,https://www.linkedin.com/jobs/view/network-aut...,automation controls engineer,'_summary'


Saved to embedding_errors.csv


## 9 — Verify: kNN search test

In [16]:
# Build a query embedding from the first row's title + skills
sample = df.iloc[0]

query_vector = np.concatenate([
    jobbert.encode(sample["_title"]),
    np.zeros(DIM),                                                         # zero-pad description slot
    jobbert.encode(" ".join(sample["_skills"]) if sample["_skills"] else "general"),
]).tolist()

response = es.search(
    index=INDEX_NAME,
    body={
        "knn": {
            "field":          "embedding",
            "query_vector":   query_vector,
            "k":              5,
            "num_candidates": 50,
        },
        "_source": ["job_id", "job_title", "skills"],
    },
)

print(f"Top 5 matches for '{sample['_title']}':")
for hit in response["hits"]["hits"]:
    src = hit["_source"]
    print(f"  [{hit['_score']:.4f}] {src['job_title']}")

Top 5 matches for 'Sr. Hardware Simulation Platform Engineer, Chassis Controls & Powertrain':
  [0.9406] Sr. Hardware Simulation Platform Engineer, Chassis Controls & Powertrain
  [0.8505] Sr. Mechanical Engineer, Vehicle & Firmware, Hardware Test Engineering
  [0.8293] Senior Software Engineer, Embedded Systems/Firmware, Platforms Infrastructure Engineering
  [0.8226] Senior System Software Engineer - Automotive AI Applications
  [0.8222] Sr. Flight Control Systems Engineer


## 10 — Index stats

In [18]:

count = es.count(index=INDEX_NAME)["count"]
print(f"Total documents in '{INDEX_NAME}': {count}")

Total documents in 'jobs': 39650
